<a href="https://colab.research.google.com/github/eshikanahata/DC-Mini-Project/blob/Nikhil/AI_DC_Task3_Nikhil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The RAG Pipeline (Chunking -> Vector Store -> Generation)

Objective:  
You will learn why "how you read" data (Chunking) matters as much as "what you read," and you will build a Retrieval Augmented Generation (RAG) pipeline.  
  



# Section 0: Setup & Prerequisites

Install the necessary libraries. You will likely need langchain, langchain-community, chromadb, and an embedding provider.

In [1]:
# INITIAL SETUP (RUN THIS ONCE)

# 1. Install dependencies with specific versions to avoid conflicts
!pip install -q -U \
  torch \
  transformers \
  sentence-transformers \
  accelerate \
  bitsandbytes \
  langchain \
  langchain-community \
  chromadb \
  pysqlite3-binary

# 2. Fix Colab's SQLite version issue (Must happen before importing chromadb)
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# 3. Check if GPU is available
import torch
if not torch.cuda.is_available():
    print("WARNING: You are running on CPU. Go to Runtime -> Change runtime type -> T4 GPU")
else:
    print(f" GPU Detected: {torch.cuda.get_device_name(0)}")

 GPU Detected: Tesla T4


In [2]:
# TODO: Import any other necessary libraries here
# !pip install ...
!pip install langchain
!pip install langchain-text-splitters

# Section 1: Chunking Experiment

LLMs have context windows. We must slice our data. But if you slice a sentence in half, the meaning might be lost. Let's prove this.

In [3]:
# Load a text file of your choice
from google.colab import files
uploaded = files.upload()

def load_data(path):
    with open(path, "r") as f:
        return f.read()

raw_text = load_data("TechNova Inc. - Company Overview.txt")
print(f"Loaded {len(raw_text)} characters.")

Saving TechNova Inc. - Company Overview.txt to TechNova Inc. - Company Overview (3).txt
Loaded 1195 characters.


Implement a splitter that strictly cuts text every x characters, regardless of sentence boundaries.

In [4]:
def naive_splitter(text, chunk_size=500):
    """
    Splits text strictly by character count.
    Returns: List[str]
    """
    # TODO: Implement strictly fixed-size splitting WITHOUT using a library (use pure Python)
    chunks = []
    for i in range(0,len(text),chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

    pass

naive_chunks = naive_splitter(raw_text)

Use a library (like LangChain) to split by "separators" (Paragraphs $\rightarrow$ Sentences $\rightarrow$ Words) to preserve meaning.

In [8]:
!pip install langchain langchain-text-splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

specific_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

semantic_chunks = specific_splitter.split_text(raw_text)
print(f"Created {len(semantic_chunks)} chunks.")

Created 4 chunks.


In [5]:
# TODO: Initialize a RecursiveCharacterTextSplitter
# Docs: Look up LangChain Text Splitters
# Constraints: Chunk size x, Chunk Overlap t


#semantic_chunks = [] # TODO: specific_splitter.split_text(raw_text)
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize a RecursiveCharacterTextSplitter
specific_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

semantic_chunks = specific_splitter.split_text(raw_text)

ModuleNotFoundError: No module named 'langchain.text_splitter'

Find a specific example where the Naive splitter broke a sentence in half, rendering it meaningless, but the Semantic splitter kept it intact

In [10]:
def find_broken_context(naive_list, semantic_list):
    """
    Print a side-by-side comparison of a specific segment where
    Naive failed and Semantic succeeded.
    """
    def find_broken_context(naive_list, semantic_list):
      """
      Print a side-by-side comparison of a specific segment where
      Naive failed and Semantic succeeded.
      """
      for i, chunk in enumerate(naive_list):
        if not chunk.rstrip().endswith(('.', '?', '!', ':', '"', "'")):
            print("NAIVE CHUNK (broken context):")
            print(chunk)
            print("\nNEAREST SEMANTIC CHUNK (for comparison):")
            # Find the semantic chunk that contains the start of this naive chunk
            target = chunk[:50]  # use the first 50 chars as a search key
            match = next((s for s in semantic_list if target[:20] in s), semantic_list[i])
            print(match)
            break

find_broken_context(naive_chunks, semantic_chunks)

In a text cell below, explain why the overlap parameter in the recursive splitter is essential for retrieval tasks.

# Section 2: Vector Storage

We will convert our semantic_chunks into vector embeddings and store them.

Initialize Embeddings & DB:
- You may use OpenAI Embeddings (if you have a key) or HuggingFace (all-MiniLM-L6-v2) for a free, local alternative.
- Use ChromaDB or FAISS as your store.

In [12]:
!pip install langchain-huggingface langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Initialize your Embedding Model
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a Vector Store from your semantic_chunks
vector_db = Chroma.from_texts(
    texts=semantic_chunks,
    embedding=embedding_function
)

print("Vector Store successfully created.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Store successfully created.


# Section 3: The MVP (Retrieval Loop)

We have the brain (LLM) and the memory (Vector DB). Now we need to wire them together.

Create a function that takes a user query, converts it to a vector, and finds the top 3 most relevant chunks from your database.

In [15]:
def retrieve_context(query, k=3):
    """
    Args:
        query (str): The user's question
        k (int): Number of chunks to retrieve
    Returns:
        List[str]: The top k context chunks
    """
    # TODO: Use your vector_db to perform a similarity search
    results = vector_db.similarity_search(query, k=k)
    return [doc.page_content for doc in results]

# Test it
test_query = ""
context_results = retrieve_context(test_query)
print(f"Retrieved {len(context_results)} chunks.")

Retrieved 3 chunks.


In [26]:
!pip install transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.5.0 which is incompatible.


Construct the final prompt. You must inject the retrieved context into the system prompt so the LLM answers only based on that data.

In [28]:
llm = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def generate_answer(query):
    context_chunks = retrieve_context(query)
    context_str = "\n".join(context_chunks)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context: {context_str}

Question: {query}

Answer:"""

    response = llm(prompt, max_new_tokens=200)[0]["generated_text"]
    return response

answer = generate_answer("Who founded TechNova?")
print(answer)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context: TechNova Inc. - Company Overview

TechNova Inc. is a software company founded in 2015 by Sarah Chen and Marcus Williams in San Francisco, California. The company specializes in cloud-based project management tools for small and medium-sized businesses.
Financials:
In 2023, TechNova reported an annual revenue of $42 million, a 35% increase from the previous year. The company raised $15 million in Series B funding in early 2024, led by Horizon Ventures.

Employees:
TechNova currently employs 320 people across offices in San Francisco, London, and Singapore. The company was ranked #4 on the "Best Startups to Work For" list by TechCrunch in 2023.
Products:
TechNova's flagship product is NovaBoard, a project management platform used by over 50,000 teams worldwide. NovaBoard allows teams to track tasks, manage deadlines, and collaborate in real time. In 2022, TechNova launched

# The End (of task 3)